In [1]:
import cv2
from PIL import Image
import numpy as np
from utils.pairing_utils import pair_gts_dets_mask

import sys
# sys.path.append("../utils")
from utils.json_parser import CellMaskDataset

from mask_rcnn_model import run_mask_rcnn
from typing import List, Tuple

ModuleNotFoundError: No module named 'utils.pairing_utils'; 'utils' is not a package

In [ ]:
ANNOTATIONS_PATH = '/home/cellareye/Cellanome/Data/analysis-images-batch-2-121422-not-reviewed/test'
IMAGES_PATH = '/home/cellareye/Cellanome/Data/Images'

In [ ]:
label_map = {1: 'Cell', 2: 'Bead', 3: 'cages'}
reverse_label_map = {value:key for key, value in label_map.items()}

# for precision and recall, there is no need to filter the annotated objects
dataset = CellMaskDataset(images_path=IMAGES_PATH, annotations_path=ANNOTATIONS_PATH,
                          # labels_of_interest = ['Cell'],
                          color_depth = 8, 
                          scale_factor = 1.0, max_larger_side=5000, max_smaller_side=5000,
                          normalize=False, class_names_to_ids_map=reverse_label_map)

In [ ]:
from cv_utils import show_detections

# colors for displaying bounding boxes
COLORS: List[Tuple[int, int, int]] = [
    (0, 0, 255),
    (255, 0, 0),
    (0, 255, 0),
    (255, 0, 255),
    (0, 255, 255),
    (255, 255, 0),
]

def show_annotations(annotations, label_map):
    
    image = annotations['image'].copy()
    # convert to 3-channels
    image = np.repeat(np.expand_dims(image, axis=2), 3, axis=2)
    
    boxes = annotations['annotations'][['xtl', 'ytl', 'xbr', 'ybr']].values
    labels  = annotations['annotations']['label'].values
    masks = annotations['masks']
    
    for i in range(len(masks)):
        # the bounding box
        (xtl, ytl, xbr, ybr) = boxes[i]
        # use a color according to the label
        color = COLORS[labels[i] % len(COLORS)]
        color_mask = color * np.repeat(np.expand_dims(masks[i][ytl:ybr, xtl:xbr], axis=2), 3, axis=2)
        blended = 0.4 * color_mask
        blended[color_mask == 0] = image[ytl:ybr, xtl:xbr][color_mask == 0]
        blended[color_mask > 0] += 0.6 * image[ytl:ybr, xtl:xbr][color_mask > 0]

        # store the blended ROI in the original image
        image[ytl:ybr, xtl:xbr] = blended.astype(np.uint8)
        
        if labels[i] in label_map:
            text = label_map[labels[i]]
        else:
            print('Incorrect ID was found %s' %labels[i])
            text = 'Unknown'
        
        # add label
        cv2.putText(image, text, (xtl, ytl + 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
        # add the bounding box with yellow color
        color = (255, 255, 0)
        cv2.rectangle(image, (xtl, ytl), (xbr, ybr), color, 1)
        
  
    # convert to PIL image to display
    return image

In [ ]:
def show_diff(dataset, idx, label_map, class_ids_of_interest=None, min_iou=0.5, post_process=False):
    """
    Args:
        dataset: Custom dataset object. 
        idx: Index of dataset.
        class_ids_of_interest (list or 1-D np.ndarray): A list of class IDs for labels to consider in 
            precision/recall evaluation. 
        min_iou (float): Minimum IoU between the mask of a ground truth and that of a detection 
            to declare a detection correct. 
        post_process (bool): If True, post-processing will be run.
    Returns
        A numpy array with differences
    """
    
        
    annots = dataset[idx]
        
    preds, _ = run_mask_rcnn(annots['image'], 
                             normalize_image = False, 
                             bit_depth = 8, 
                             crop = True, 
                             post_process = post_process, 
                             plot_results = False)
    
        
    num_true_positives: int = 0
    num_false_positives: int = 0
    num_false_negatives: int = 0
        
    boxes = np.array(preds['boxes']) if len(preds['boxes']) > 0 else np.zeros((0, 4), dtype=int)
    labels = np.array(preds['labels']) if len(preds['labels']) > 0 else np.zeros((0,), dtype=int)
    scores = np.array(preds['scores']) if len(preds['scores']) > 0 else np.zeros((0,), dtype=float)
    masks = preds['masks']
        
    if class_ids_of_interest is None:
        # use the union of all the class IDs from the detections and the annotations
        class_ids_to_filter = list(annots['annotations']['label'].unique())
        class_ids_to_filter += list(np.unique(labels))
        # remove duplicates
        class_ids_to_filter = list(set(class_ids_to_filter))
    else:
        class_ids_to_filter = class_ids_of_interest
    
    fn_idxs: List[int] = []
    fp_idxs: List[int] = []
    
    for class_id in class_ids_to_filter:
        # filter the detections and ground truths for the given label
        idxs = annots['annotations'][annots['annotations']['label'] == class_id].index.values
        gt_boxes = annots['annotations'].loc[idxs, ['xtl', 'ytl', 'xbr', 'ybr']].values.astype(int)
        gt_masks = [annots['masks'][ind][gt_boxes[i][1]:gt_boxes[i][3], gt_boxes[i][0]:gt_boxes[i][2]] 
                    for i, ind in enumerate(idxs)]
            
        det_boxes = boxes[labels == class_id, :]
        det_masks = [mask for i, mask in enumerate(masks) if labels[i] == class_id]
        # pair
        paired_idx, unpaired_gts, unpaired_dets = pair_gts_dets_mask(gt_boxes, gt_masks, det_boxes, 
                                                                     det_masks, min_iou)
        
        fn_idxs += [idxs[i] for i in unpaired_gts]
        
        temp_idxs = np.where(labels == class_id)[0]
        fp_idxs += [temp_idxs[i] for i in unpaired_dets]
        
        num_true_positives += len(paired_idx)
        num_false_positives += len(unpaired_dets)
        num_false_negatives += len(unpaired_gts)
        
    precision = num_true_positives / (num_true_positives + num_false_positives + 1e-30)
    recall = num_true_positives / (num_true_positives + num_false_negatives + 1e-30)
    
    fn_annotations = {}
    fn_annotations['image'] = annots['image'] 
    fn_annotations['annotations'] = annots['annotations'].loc[fn_idxs].copy().reset_index(drop=True)
    fn_annotations['masks'] = [annots['masks'][i] for i in fn_idxs]
    
    fp_detections = {}
    fp_detections['boxes'] = [box for i, box in enumerate(boxes) if i in fp_idxs]
    fp_detections['labels'] = [label for i, label in enumerate(labels) if i in fp_idxs]
    fp_detections['scores'] = [score for i, score in enumerate(scores) if i in fp_idxs]
    fp_detections['masks'] = [mask for i, mask in enumerate(masks) if i in fp_idxs]
    
    img = show_annotations(fn_annotations, label_map)
    img = show_detections(img, fp_detections, label_map)
        
    return fn_annotations, fp_detections, img

In [ ]:
idx = 9
fn_annotations, fp_detections, img = show_diff(dataset, idx, label_map,
                                               class_ids_of_interest=[reverse_label_map['Cell']], 
                                               min_iou=0.5, post_process=False)
Image.fromarray(img)